# 인공지능(CNN) - 위클리 챌린지

In [45]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

## Phase 1

- Tabular 데이터셋 Titianic 사용
    1. (공통) 데이터셋 생성
    2. (전처리) 학습, 검증, 테스트 데이터셋 분할하기
    3. (전통 머신러닝) KNN 알고리즘으로 학습 및 예측하기
    4. (전통 머신러닝) Perceptron, SVM, Random Forest, Native Bayes로 학습하기

In [ ]:
# 1. 데이터셋 생성 - 수집
import kagglehub

# Download latest version
path = kagglehub.dataset_download("euclidsoft/titianic")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'titianic' dataset.
Path to dataset files: /kaggle/input/titianic


In [ ]:
# 1.1. 데이터셋 생성 - 호출
import os

excel_path = os.path.join(path, "titanic.xls")
df = pd.read_excel(excel_path)
df

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.00,0,0,24160,211.34,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.92,1,2,113781,151.55,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.00,1,2,113781,151.55,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.00,1,2,113781,151.55,C22 C26,S,NaN,135.00,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.00,1,2,113781,151.55,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,3,0,"Zabour, Miss. Hileni",female,14.50,1,0,2665,14.45,NaN,C,NaN,328.00,NaN
1305,3,0,"Zabour, Miss. Thamine",female,NaN,1,0,2665,14.45,NaN,C,NaN,NaN,NaN
1306,3,0,"Zakarian, Mr. Mapriededer",male,26.50,0,0,2656,7.22,NaN,C,NaN,304.00,NaN
1307,3,0,"Zakarian, Mr. Ortin",male,27.00,0,0,2670,7.22,NaN,C,NaN,NaN,NaN


In [40]:
# 1.2. 데이터셋 X, y 분리하기 (테스트할 인과관계 분리하기)
X = df.drop('survived', axis=1)
y = df['survived']

In [55]:
# 2. (데이터 전처리) 학습, 검증, 테스트 데이터셋 분할하기
    # 분할 기법: 직접 분할, 랜덤 분할

# 2.1. 직접 분할 기법
# (X) Train Data
TRAIN_RATIO = 0.6
X_train_size = int(TRAIN_RATIO * len(X))
X_train_data = df[:X_train_size]

# (X) Validation Data
VALIDATION_RATIO = 0.2
X_validation_size = int(VALIDATION_RATIO * len(X))
X_validation_data = df[X_train_size:X_train_size + X_validation_size]

# (X) Test Data
X_test_data = df[X_train_size + X_validation_size:]

print("X 학습 데이터셋 크기: ", len(X_train_data))
print("X 검증 데이터셋 크기: ", len(X_validation_data))
print("X 테스트 데이터셋 크기: ", len(X_test_data))


# (y) Train Data
TRAIN_RATIO = 0.6
y_train_size = int(TRAIN_RATIO * len(y))
y_train_data = df[:y_train_size]

# (y) Validation Data
VALIDATION_RATIO = 0.2
y_validation_size = int(VALIDATION_RATIO * len(y))
y_validation_data = df[y_train_size:y_train_size + y_validation_size]

# (y) Test Data
y_test_data = df[y_train_size + y_validation_size:]

print("y 학습 데이터셋 크기: ", len(y_train_data))
print("y 검증 데이터셋 크기: ", len(y_validation_data))
print("y 테스트 데이터셋 크기: ", len(y_test_data))

X 학습 데이터셋 크기:  785
X 검증 데이터셋 크기:  261
X 테스트 데이터셋 크기:  263
y 학습 데이터셋 크기:  785
y 검증 데이터셋 크기:  261
y 테스트 데이터셋 크기:  263


In [68]:
# 2.2. 랜덤 분할 기법
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
X_train_data, X_temp_data, y_train_data, y_temp_data = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)
X_validation_data, X_test_data, y_validation_data, y_test_data = train_test_split(
    X_temp_data, y_temp_data, test_size=0.5, stratify=y_temp_data, random_state=42
)

print("X 학습 데이터셋 크기: ", len(X_train_data))
print("X 검증 데이터셋 크기: ", len(X_validation_data))
print("X 테스트 데이터셋 크기: ", len(X_test_data))

print("학습 데이터셋 크기: ", len(y_train_data))
print("검증 데이터셋 크기: ", len(y_validation_data))
print("테스트 데이터셋 크기: ", len(y_test_data))

print(X_train_data.columns.tolist())

X 학습 데이터셋 크기:  785
X 검증 데이터셋 크기:  262
X 테스트 데이터셋 크기:  262
학습 데이터셋 크기:  785
검증 데이터셋 크기:  262
테스트 데이터셋 크기:  262
['pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest']


In [61]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# ============================================
# 1. 사용할 컬럼 정의 (중요!)
# ============================================
numeric_features = ['pclass', 'age', 'sibsp', 'parch', 'fare']
categorical_features = ['sex', 'embarked']

# ============================================
# 2. 전처리 파이프라인 구성
# ============================================
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [74]:
# 3. (전통 머신러닝) KNN 알고리즘으로 학습 및 예측하기
from sklearn.preprocessing import StandardScaler

# 3.1. 데이터 정규화 (거리 기반 알고리즘이라 스케일 조정이 중요)
X_train_scaled = preprocessor.fit_transform(X_train_data)
X_val_scaled = preprocessor.transform(X_validation_data)
X_test_scaled = preprocessor.transform(X_test_data)

y_train = y_train_data.values
y_val = y_validation_data.values
y_test = y_test_data.values

# 3.2. PyTorch Tensor로 변환 (X, KNN이므로 tensor 필요 없음)

In [73]:
from sklearn.neighbors import KNeighborsClassifier
# 3.3. K-NN 모델 생성 (K=5 설정)
k = 5
knn_model = KNeighborsClassifier(n_neighbors=k)

# 3.4. 모델 학습
knn_model.fit(X_train_scaled, y_train_data.values.ravel())

KNeighborsClassifier()

In [71]:
# 3.5. 모델 평가
val_accuracy = knn_model.score(X_val_scaled, y_validation_data.values.ravel())
test_accuracy = knn_model.score(X_test_scaled, y_test_data.values.ravel())

print(f"Validation 데이터 정확도: {val_accuracy:.4f}")
print(f"Test 데이터 정확도: {test_accuracy:.4f}")

# 3.6. 샘플 데이터 예측 (첫 번째 테스트 샘플 예측)
sample_idx = 1
sample = X_test_scaled[sample_idx].reshape(1, -1)  # 입력 데이터 형태 맞추기
prediction = knn_model.predict(sample)

print(f"실제값: {y_test_data.iloc[sample_idx]}, 예측값: {prediction[0]}")

Validation 데이터 정확도: 0.5992
Test 데이터 정확도: 0.5267
실제값: 1, 예측값: 0


In [76]:
# 4. (전통 머신러닝) Perceptron, SVM, Random Forest, Naive Bayes로 학습하기
from sklearn.linear_model import Perceptron
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

# ==================== 모델 정의 ====================
models = {
    'Perceptron': Perceptron(max_iter=1000, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Naive Bayes': GaussianNB()
}

results = []

for name, model in models.items():
    
    # y를 1차원 numpy array로 변환 (안전하게 처리)
    y_train_1d = np.asarray(y_train).ravel()
    y_val_1d   = np.asarray(y_val).ravel()
    y_test_1d  = np.asarray(y_test).ravel()
    
    # 모델 학습
    model.fit(X_train_scaled, y_train_1d)
    
    # Validation 성능
    val_pred = model.predict(X_val_scaled)
    val_acc = accuracy_score(y_val_1d, val_pred)
    
    # Test 성능
    test_pred = model.predict(X_test_scaled)
    test_acc = accuracy_score(y_test_1d, test_pred)
    
    results.append({
        'Model': name,
        'Validation Accuracy': round(val_acc, 4),
        'Test Accuracy': round(test_acc, 4)
    })
    
    print(f"{name:15} | Validation: {val_acc:.4f} | Test: {test_acc:.4f}")

# 결과 테이블 출력
results_df = pd.DataFrame(results)
print("\n=== 모델 성능 비교 (Validation 기준 정렬) ===")
print(results_df.sort_values('Validation Accuracy', ascending=False).to_string(index=False))

Perceptron      | Validation: 0.6260 | Test: 0.6756
SVM             | Validation: 0.8015 | Test: 0.8435
Random Forest   | Validation: 0.7786 | Test: 0.7939
Naive Bayes     | Validation: 0.7634 | Test: 0.8015

=== 모델 성능 비교 (Validation 기준 정렬) ===
        Model  Validation Accuracy  Test Accuracy
          SVM                 0.80           0.84
Random Forest                 0.78           0.79
  Naive Bayes                 0.76           0.80
   Perceptron                 0.63           0.68


## Phase 2

- CNN

In [ ]:
# 2. (데이터 전처리) 증강(Data Augmentation)기법 적용 및 비교하기